# Experimental comparison of tabular classifiers

This notebook implements a reproducible binary classification study using two real datasets. The workflow covers data description, exploratory analysis, leakage-safe preprocessing, compact model selection, 21 paired repeated evaluations, final-test diagnostics, interpretability, and paired statistical comparison.

The generated tables remain inside the notebook. Only a compact set of analytical figures is saved for later use in the scientific article and presentation.

In [ ]:
from __future__ import annotations

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import sys

from pathlib import Path


from IPython.display import display

def find_project_root(start: Path) -> Path:
    """Find the project root from the current Jupyter working directory."""

    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "classification_benchmark").is_dir() and (candidate / "requirements.txt").exists():
            return candidate
    raise RuntimeError("Project root not found. Start Jupyter from the extracted project directory.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from classification_benchmark.config import ExperimentConfig
from classification_benchmark.datasets import DatasetBundle, load_dataset
from classification_benchmark.evaluation import (
    build_reference_prediction_frame,
    evaluate_repeated_runs,
    final_test_metrics,
    fit_reference_models,
    summarize_results,
)
from classification_benchmark.io import ensure_figure_directory
from classification_benchmark.metrics import ks_curve_data
from classification_benchmark.models import default_model_specs
from classification_benchmark.selection import select_configurations
from classification_benchmark.statistics import cochran_q_tests, pairwise_mcnemar_tests
from classification_benchmark.visualization import (
    BLUE,
    DEEP_BLUE,
    DEEP_GREEN,
    GREEN,
    LIGHT_BLUE,
    LIGHT_GREEN,
    PALETTE,
    TEAL,
    plot_decision_tree_model,
    plot_discrimination_curves,
    plot_operating_point_diagnostics,
    plot_performance_overview,
    plot_permutation_importance,
    plot_ranking_diagnostics,
)

In [ ]:
%load_ext watermark
%watermark -v -m -p numpy,pandas,scipy,sklearn,matplotlib,joblib

## 1. Experiment configuration

The final protocol uses two real datasets, four complementary classifiers, one validation-based configuration stage, 21 paired development evaluations, and one isolated final test set. Model configurations and decision thresholds are selected before the repeated and final evaluations.

In [ ]:
config = ExperimentConfig(
    dataset_names=("credit_risk_openml", "bank_marketing_openml"),
    n_runs=21,
    test_size=0.20,
    validation_size=0.20,
    random_seed=42,
    output_dir=PROJECT_ROOT / "outputs",
    cache_dir=PROJECT_ROOT / "data" / "cache",
)
config.validate()
figure_dir = ensure_figure_directory(config.output_dir)

plt.rcParams.update(
    {
        "figure.figsize": (9, 5),
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "font.size": 10,
        "axes.grid": False,
    }
)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 2. Dataset loading and validation

Both datasets are loaded from OpenML through `sklearn.datasets.fetch_openml`. The original sources are the UCI Machine Learning Repository datasets documented in the references file.

- **German Credit Risk:** the positive class is `bad`.
- **Bank Marketing:** the positive class is `yes`. The `duration` predictor is excluded because it is only known after the campaign call and would not be available in a pre-contact targeting scenario.

The loaders validate the expected class mapping and positive-class rate before the experiment proceeds.

In [ ]:
%%time

datasets = [load_dataset(name, config.cache_dir) for name in config.dataset_names]

dataset_overview = pd.DataFrame(
    [
        {
            "dataset": dataset.name,
            "display_name": dataset.display_name,
            "description": dataset.description,
            "source": dataset.source,
            "source_url": dataset.source_url,
            "records": dataset.n_samples,
            "predictors": dataset.n_features,
            "positive_class": dataset.positive_class,
            "positive_rate": dataset.y.mean(),
            "excluded_features": ", ".join(dataset.excluded_features) or "None",
        }
        for dataset in datasets
    ]
)
display(dataset_overview)

## 3. Data dictionaries

The dictionaries document the meaning of every predictor retained in the modeling datasets. They also report the observed type, number of unique values, and missing-value count.

In [ ]:
DATA_DICTIONARIES = {
    "credit_risk_openml": {
        "checking_status": "Status of the existing checking account.",
        "duration": "Credit duration in months.",
        "credit_history": "Summary of the applicant's previous credit history.",
        "purpose": "Declared purpose of the requested credit.",
        "credit_amount": "Requested credit amount.",
        "savings_status": "Savings account or bond status.",
        "employment": "Length or category of current employment.",
        "installment_commitment": "Installment rate as a percentage of disposable income.",
        "personal_status": "Personal status and sex category.",
        "other_parties": "Other debtors or guarantors.",
        "residence_since": "Years at the current residence.",
        "property_magnitude": "Property category associated with the applicant.",
        "age": "Applicant age in years.",
        "other_payment_plans": "Other installment plans.",
        "housing": "Housing situation.",
        "existing_credits": "Number of existing credits at the institution.",
        "job": "Employment or job category.",
        "num_dependents": "Number of people financially supported.",
        "own_telephone": "Telephone ownership indicator.",
        "foreign_worker": "Foreign worker indicator.",
    },
    "bank_marketing_openml": {
        "age": "Customer age in years.",
        "job": "Customer occupation category.",
        "marital": "Marital status.",
        "education": "Education category.",
        "default": "Credit default indicator.",
        "balance": "Average yearly account balance.",
        "housing": "Housing loan indicator.",
        "loan": "Personal loan indicator.",
        "contact": "Communication channel used in the campaign.",
        "day": "Day of the month of the last contact.",
        "month": "Month of the last contact.",
        "campaign": "Number of contacts during the current campaign.",
        "pdays": "Days since the customer was previously contacted; -1 means no previous contact.",
        "previous": "Number of contacts before the current campaign.",
        "poutcome": "Outcome of the previous marketing campaign.",
    },
}

def build_data_dictionary(dataset: DatasetBundle) -> pd.DataFrame:
    """Build a dataset-specific data dictionary for notebook inspection."""

    descriptions = DATA_DICTIONARIES[dataset.name]
    return pd.DataFrame(
        [
            {
                "feature": column,
                "description": descriptions[column],
                "dtype": str(dataset.X[column].dtype),
                "unique_values": int(dataset.X[column].nunique(dropna=True)),
                "missing_values": int(dataset.X[column].isna().sum()),
            }
            for column in dataset.X.columns
        ]
    )


for dataset in datasets:
    print(f"\n{dataset.display_name}")
    display(build_data_dictionary(dataset))

## 4. Exploratory data analysis

The EDA verifies records, predictors, types, duplicates, missing values, class balance, descriptive statistics, and extreme numeric values. Observations below the 1st percentile or above the 99th percentile are reported as percentile extremes. They are not deleted from the raw dataset.

For preprocessing, percentile clipping is applied only to Logistic Regression and its limits are learned exclusively from the training data. Tree-based models retain the original numeric scale because they are not sensitive to monotonic scaling and are less affected by extreme values.

In [ ]:
def percentile_extreme_report(frame: pd.DataFrame) -> pd.DataFrame:
    """Summarize numeric observations outside the 1st and 99th percentiles."""

    numeric = frame.select_dtypes(include=["number", "bool"])
    rows = []
    for column in numeric.columns:
        series = pd.to_numeric(numeric[column], errors="coerce")
        lower = series.quantile(0.01)
        upper = series.quantile(0.99)
        below = int((series < lower).sum())
        above = int((series > upper).sum())
        rows.append(
            {
                "feature": column,
                "p01": lower,
                "p99": upper,
                "below_p01": below,
                "above_p99": above,
                "extreme_count": below + above,
                "extreme_rate": (below + above) / max(series.notna().sum(), 1),
            }
        )
    return pd.DataFrame(rows).sort_values("extreme_rate", ascending=False)

def show_and_close(fig: plt.Figure) -> None:
    """Display a figure and release its Matplotlib resources."""

    fig.tight_layout()
    plt.show()
    plt.close(fig)

def save_show_and_close(fig: plt.Figure, filename: str) -> None:
    """Save one selected EDA figure, display it, and close it."""

    destination = figure_dir / filename
    fig.tight_layout()
    fig.savefig(destination, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

def show_dataset_eda(dataset: DatasetBundle) -> None:
    """Display the notebook EDA and save only the compact overview figure."""

    print("\n" + "=" * 90)
    print(dataset.display_name)
    print("=" * 90)
    display(dataset.X.head())

    quality_summary = pd.DataFrame(
        {
            "measure": [
                "records",
                "predictors",
                "numeric_predictors",
                "categorical_predictors",
                "missing_cells",
                "duplicate_rows",
                "constant_predictors",
                "positive_class_rate",
            ],
            "value": [
                dataset.n_samples,
                dataset.n_features,
                dataset.X.select_dtypes(include=["number", "bool"]).shape[1],
                dataset.X.select_dtypes(exclude=["number", "bool"]).shape[1],
                int(dataset.X.isna().sum().sum()),
                int(dataset.X.duplicated().sum()),
                int((dataset.X.nunique(dropna=False) <= 1).sum()),
                float(dataset.y.mean()),
            ],
        }
    )
    display(quality_summary)

    missing_summary = (
        dataset.X.isna()
        .sum()
        .rename("missing_count")
        .to_frame()
        .assign(missing_rate=lambda frame: frame["missing_count"] / dataset.n_samples)
        .sort_values("missing_rate", ascending=False)
    )
    display(missing_summary)

    class_summary = (
        dataset.y.value_counts(dropna=False)
        .sort_index()
        .rename("count")
        .to_frame()
        .assign(rate=lambda frame: frame["count"] / frame["count"].sum())
    )
    class_summary.index = dataset.target_names
    display(class_summary)

    numeric = dataset.X.select_dtypes(include=["number", "bool"])
    descriptive = numeric.describe(
        percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
    )
    display(descriptive)

    extremes = percentile_extreme_report(dataset.X)
    display(extremes)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
    axes[0].bar(
        class_summary.index,
        class_summary["count"],
        color=[LIGHT_BLUE, TEAL],
        alpha=0.90,
    )
    axes[0].set_title("Class distribution")
    axes[0].set_xlabel("Class")
    axes[0].set_ylabel("Records")
    axes[0].grid(axis="y", alpha=0.20)

    extreme_plot = extremes.head(10).sort_values("extreme_rate")
    axes[1].barh(
        extreme_plot["feature"],
        extreme_plot["extreme_rate"],
        color=TEAL,
        alpha=0.85,
    )
    axes[1].set_title("Extreme-value rate outside P01-P99")
    axes[1].set_xlabel("Rate")
    axes[1].set_ylabel("Feature")
    axes[1].grid(axis="x", alpha=0.20)
    fig.suptitle(f"EDA overview - {dataset.display_name}", y=1.02)
    save_show_and_close(fig, f"{dataset.name}_eda_overview.png")

    selected_numeric = numeric.columns[: min(6, numeric.shape[1])].tolist()
    rows = int(np.ceil(len(selected_numeric) / 2))
    fig, axes = plt.subplots(rows, 2, figsize=(12, 3.5 * rows))
    axes_array = np.atleast_1d(axes).ravel()
    for index, column in enumerate(selected_numeric):
        axes_array[index].hist(
            numeric[column].dropna(),
            bins=25,
            color=PALETTE[index % len(PALETTE)],
            alpha=0.80,
        )
        axes_array[index].set_title(column)
        axes_array[index].set_xlabel("Value")
        axes_array[index].set_ylabel("Frequency")
        axes_array[index].grid(axis="y", alpha=0.15)
    for index in range(len(selected_numeric), len(axes_array)):
        axes_array[index].axis("off")
    fig.suptitle(f"Numeric distributions - {dataset.display_name}", y=1.01)
    show_and_close(fig)

    fig, axes = plt.subplots(rows, 2, figsize=(12, 3.2 * rows))
    axes_array = np.atleast_1d(axes).ravel()
    for index, column in enumerate(selected_numeric):
        box = axes_array[index].boxplot(
            numeric[column].dropna().to_numpy(),
            vert=False,
            patch_artist=True,
            medianprops={"color": DEEP_BLUE, "linewidth": 1.8},
            boxprops={"facecolor": PALETTE[index % len(PALETTE)], "alpha": 0.70},
            whiskerprops={"color": DEEP_GREEN},
            capprops={"color": DEEP_GREEN},
        )
        axes_array[index].set_title(column)
        axes_array[index].set_xlabel("Value")
        axes_array[index].grid(axis="x", alpha=0.15)
    for index in range(len(selected_numeric), len(axes_array)):
        axes_array[index].axis("off")
    fig.suptitle(f"Numeric boxplots - {dataset.display_name}", y=1.01)
    show_and_close(fig)

    correlation = numeric.corr(numeric_only=True)
    if correlation.shape[0] > 1:
        fig, ax = plt.subplots(figsize=(10, 8))
        image = ax.imshow(correlation, cmap="GnBu", vmin=-1.0, vmax=1.0)
        ax.set_xticks(
            range(len(correlation.columns)),
            labels=correlation.columns,
            rotation=90,
            fontsize=8,
        )
        ax.set_yticks(
            range(len(correlation.index)),
            labels=correlation.index,
            fontsize=8,
        )
        ax.set_title(f"Numeric correlation matrix - {dataset.display_name}")
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
        show_and_close(fig)

for dataset in datasets:
    show_dataset_eda(dataset)

## 5. Preprocessing strategy

All transformations are contained in scikit-learn pipelines to prevent data leakage.

- Numeric missing values are imputed with medians learned from training data.
- Logistic Regression receives P01-P99 clipping and standardization learned from training data.
- Tree-based models retain the original numeric scale and do not receive clipping.
- Categorical missing values are imputed with the training mode.
- Categorical variables are one-hot encoded while unseen categories are ignored.

In [ ]:
preprocessing_summary = pd.DataFrame(
    [
        {
            "model_family": "All models",
            "step": "Median/mode imputation",
            "purpose": "Handle missing values using training information only.",
        },
        {
            "model_family": "Logistic Regression",
            "step": "P01-P99 clipping",
            "purpose": "Limit extreme numeric influence using training percentiles.",
        },
        {
            "model_family": "Logistic Regression",
            "step": "Standardization",
            "purpose": "Place numeric predictors on comparable scales.",
        },
        {
            "model_family": "Tree-based models",
            "step": "Original numeric scale",
            "purpose": "Preserve natural split thresholds without unnecessary scaling or clipping.",
        },
        {
            "model_family": "All models",
            "step": "One-hot encoding",
            "purpose": "Convert categorical predictors into model-compatible numeric columns.",
        },
    ]
)
display(preprocessing_summary)

## 6. Algorithms and compact parameter selection

The comparison includes a linear probabilistic baseline, an interpretable single tree, a bagging ensemble, and a boosting model. Each model evaluates three predefined configurations on the same stratified validation split.

Average Precision is the primary configuration criterion because both tasks contain a minority positive class. Validation F1-score and ROC-AUC are used as tie-breakers. The decision threshold is selected by validation F1-score.

In [ ]:
model_specs = default_model_specs()

model_description = pd.DataFrame(
    [
        {
            "model": "Logistic Regression",
            "family": "Linear probabilistic classifier",
            "role": "Interpretable regularized baseline.",
        },
        {
            "model": "Decision Tree",
            "family": "Rule-based tree",
            "role": "Directly interpretable nonlinear classifier.",
        },
        {
            "model": "Random Forest",
            "family": "Bagging ensemble",
            "role": "Reduces individual-tree variance through bootstrap aggregation.",
        },
        {
            "model": "Histogram Gradient Boosting",
            "family": "Boosting ensemble",
            "role": "Builds sequential trees to capture nonlinear interactions.",
        },
    ]
)
display(model_description)

candidate_table = pd.DataFrame(
    [
        {
            "model": model_spec.display_name,
            "candidate": candidate_index,
            "parameters": parameters,
        }
        for model_spec in model_specs
        for candidate_index, parameters in enumerate(model_spec.candidate_parameters, start=1)
    ]
)
display(candidate_table)

In [ ]:
%%time

selection_results, selected_configurations = select_configurations(
    datasets,
    model_specs,
    config,
)

selected_rows = []
for (dataset_name, model_name), selected in selected_configurations.items():
    model_display_name = next(
        model_spec.display_name for model_spec in model_specs if model_spec.name == model_name
    )
    selected_rows.append(
        {
            "dataset": dataset_name,
            "model": model_display_name,
            "parameters": selected.parameters,
            "threshold": selected.threshold,
            "validation_f1": selected.validation_f1,
            "validation_average_precision": selected.validation_average_precision,
            "validation_roc_auc": selected.validation_roc_auc,
        }
    )
selected_table = pd.DataFrame(selected_rows).sort_values(["dataset", "model"])
display(selected_table)

## 7. Repeated development evaluation

The selected configurations and thresholds remain fixed. The models are then evaluated on 21 paired stratified train-validation partitions drawn from the development set. These repetitions describe performance stability through means, standard deviations, and distributions; they are not treated as independent samples for final inferential claims.

In [ ]:
%%time

repeated_results = evaluate_repeated_runs(
    datasets,
    model_specs,
    selected_configurations,
    config,
)
repeated_summary = summarize_results(repeated_results)

summary_columns = [
    "dataset_display_name",
    "model_display_name",
    "f1_mean",
    "f1_std",
    "roc_auc_mean",
    "roc_auc_std",
    "average_precision_mean",
    "average_precision_std",
    "ks_statistic_mean",
    "ks_statistic_std",
]
display(
    repeated_summary[summary_columns].sort_values(
        ["dataset_display_name", "f1_mean"],
        ascending=[True, False],
    )
)

best_model_names = (
    repeated_summary.sort_values(["dataset", "f1_mean"], ascending=[True, False])
    .groupby("dataset", as_index=False)
    .first()
    .set_index("dataset")["model"]
    .to_dict()
)
best_model_names

## 8. Isolated final-test evaluation

Every selected model is fitted on the complete development partition and evaluated once on the fixed 20% test partition. The test set is not used for preprocessing, parameter selection, threshold selection, or selection of the model used in the detailed analytical plots.

In [ ]:
%%time

reference_models = fit_reference_models(
    datasets,
    model_specs,
    selected_configurations,
    config,
)
final_metrics = final_test_metrics(reference_models)

display(
    final_metrics[
        [
            "dataset_display_name",
            "model_display_name",
            "threshold",
            "balanced_accuracy",
            "precision",
            "recall",
            "specificity",
            "f1",
            "roc_auc",
            "average_precision",
            "ks_statistic",
            "ks_pvalue",
        ]
    ].sort_values(["dataset_display_name", "f1"], ascending=[True, False])
)

## 9. Paired classifier comparison on the final test set

Cochran's Q compares the correctness of all classifiers on the same final-test observations. When the global test is significant, exact pairwise McNemar tests identify which classifier pairs differ. Holm correction controls the family-wise error rate across the pairwise comparisons.

This design compares paired predictions on the isolated test set and avoids treating overlapping repeated splits as independent observations.

In [ ]:
reference_predictions = build_reference_prediction_frame(reference_models)
cochran_results = cochran_q_tests(reference_predictions)
mcnemar_results = pairwise_mcnemar_tests(reference_predictions)

def format_p_value(value: float) -> str:
    """Format small p-values without reporting them as zero."""

    return "< 0.0001" if value < 0.0001 else f"{value:.4f}"

cochran_display = cochran_results.copy()
cochran_display["p_value"] = cochran_display["p_value"].map(format_p_value)
display(cochran_display)

mcnemar_display = mcnemar_results[
    [
        "dataset",
        "model_a_display_name",
        "model_b_display_name",
        "a_correct_b_wrong",
        "a_wrong_b_correct",
        "discordant_effect",
        "p_value_holm",
        "significant_0_05",
    ]
].copy()
mcnemar_display["p_value_holm"] = mcnemar_display["p_value_holm"].map(format_p_value)
display(mcnemar_display)

## 10. Selected analytical figures

Only a compact set of figures is saved:

1. EDA overview;
2. repeated F1 performance and stability;
3. ROC and precision-recall curves;
4. confusion matrix and threshold trade-off for the development-selected model;
5. cumulative gains and Kolmogorov-Smirnov curves;
6. permutation importance;
7. the first three levels of the Decision Tree.

All figures use blue and green palettes.

In [ ]:
%%time

def plot_ranking_diagnostics(reference, path):
    population_frame = pd.DataFrame(
        {
            "target": reference.y_test.to_numpy(),
            "score": reference.scores.to_numpy(),
        }
    ).sort_values("score", ascending=False, ignore_index=True)

    population = np.arange(1, len(population_frame) + 1) / len(population_frame)
    positive_total = max(int(population_frame["target"].sum()), 1)
    gains = population_frame["target"].cumsum().to_numpy() / positive_total

    thresholds, positive_cdf, negative_cdf, ks_value, ks_threshold = ks_curve_data(
        reference.y_test.to_numpy(),
        reference.scores.to_numpy(),
    )
    threshold_index = int(np.argmin(np.abs(thresholds - ks_threshold)))

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(
        population,
        gains,
        color=BLUE,
        linewidth=2,
        label=reference.model_spec.display_name,
    )
    axes[0].plot(
        [0, 1],
        [0, 1],
        linestyle="--",
        color=LIGHT_GREEN,
        label="Random selection",
    )
    axes[0].set_xlabel("Population proportion")
    axes[0].set_ylabel("Positive cases captured")
    axes[0].set_title("Cumulative gains")
    axes[0].legend(fontsize=8)
    axes[0].grid(alpha=0.20)

    axes[1].plot(
        thresholds,
        negative_cdf,
        color=BLUE,
        linewidth=2,
        label=reference.dataset.target_names[0],
    )
    axes[1].plot(
        thresholds,
        positive_cdf,
        color=GREEN,
        linewidth=2,
        label=reference.dataset.target_names[1],
    )
    axes[1].vlines(
        ks_threshold,
        positive_cdf[threshold_index],
        negative_cdf[threshold_index],
        color=TEAL,
        linestyle="--",
        linewidth=2,
        label=f"KS={ks_value:.3f} at {ks_threshold:.3f}",
    )
    axes[1].set_xlabel("Predicted probability threshold")
    axes[1].set_ylabel("Empirical cumulative proportion")
    axes[1].set_title("Kolmogorov-Smirnov curve")
    axes[1].legend(fontsize=8)
    axes[1].grid(alpha=0.20)

    fig.suptitle(
        f"Ranking diagnostics - {reference.model_spec.display_name}",
        y=1.03,
        fontsize=14,
    )
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)

ranking_output_names = {
    "credit_risk_openml": "german_credit_ks_cumulative_gain.png",
    "bank_marketing_openml": "bank_marketing_ks_cumulative_gain.png",
}

importance_tables = {}

for dataset in datasets:
    dataset_models = {
        model_spec.name: reference_models[(dataset.name, model_spec.name)]
        for model_spec in model_specs
    }

    best_reference = reference_models[
        (dataset.name, best_model_names[dataset.name])
    ]
    decision_tree_reference = reference_models[
        (dataset.name, "decision_tree")
    ]

    plot_performance_overview(
        repeated_summary,
        repeated_results,
        dataset.name,
        figure_dir / f"{dataset.name}_performance_overview.png",
    )

    plot_discrimination_curves(
        dataset_models,
        figure_dir / f"{dataset.name}_discrimination_curves.png",
    )

    plot_operating_point_diagnostics(
        best_reference,
        figure_dir / f"{dataset.name}_operating_point.png",
    )

    plot_ranking_diagnostics(
        best_reference,
        figure_dir / ranking_output_names[dataset.name],
    )

    _, importance = plot_permutation_importance(
        best_reference,
        figure_dir / f"{dataset.name}_permutation_importance.png",
    )

    importance_tables[dataset.name] = importance

    plot_decision_tree_model(
        decision_tree_reference,
        figure_dir / f"{dataset.name}_decision_tree.png",
        display_depth=3,
    )

for dataset in datasets:
    print(f"\nTop permutation importance - {dataset.display_name}")
    display(importance_tables[dataset.name].head(12))

print(f"Selected figures saved to: {figure_dir}")